# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)

This notebook demonstrates probabilistic patient linkage using Splink across disparate EHR records (Fellegi-Sunter model).

In [ ]:
import duckdb
import pandas as pd
from splink.duckdb.linker import DuckDBLinker
from splink.duckdb.blocking_rule_library import block_on
import splink.duckdb.comparison_library as cl
import splink.duckdb.comparison_template_library as ctl

print("Libraries loaded.")

In [ ]:
# Create synthetic disparate EHR records
data = [
    {"unique_id": "A1", "first_name": "John", "last_name": "Doe", "dob": "1980-01-01", "city": "New York"},
    {"unique_id": "B1", "first_name": "Jon", "last_name": "Doe", "dob": "1980-01-01", "city": "New York"},
    {"unique_id": "C1", "first_name": "Jane", "last_name": "Smith", "dob": "1990-05-15", "city": "Boston"}
]
df = pd.DataFrame(data)
print("Raw Patient Records:")
print(df)

In [ ]:
# Define Splink Linkage Settings
settings = {
    "link_type": "dedupe_only",
    "blocking_rules_to_generate_predictions": [
        block_on("dob"),
        block_on("city")
    ],
    "comparisons": [
        ctl.name_comparison("first_name"),
        ctl.name_comparison("last_name"),
        ctl.date_comparison("dob", cast_strings_to_date=True),
        cl.exact_match("city", term_frequency_adjustments=True)
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True,
}

# Initialize DuckDB Linker
linker = DuckDBLinker(df, settings)

# Estimate model parameters (in a real scenario with more data, you'd use EM training)
linker.estimate_u_using_random_sampling(max_pairs=1e6)
linker.estimate_parameters_using_expectation_maximisation(block_on("dob"))

# Predict matches
df_predictions = linker.predict(threshold_match_probability=0.5)

print("\nProbabilistic Match Predictions:")
print(df_predictions.as_pandas_dataframe()[["unique_id_l", "unique_id_r", "match_probability"]])